<a href="https://colab.research.google.com/github/Prina5/lab-4-llm-decision-support/blob/main/lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# API-key setup — DO NOT hard-code your key in this cell.
import json
import os
import pandas as pd
from openai import OpenAI
from google.colab import userdata



# --- Google Colab (Secrets panel) ---
# from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
    )
MODEL = "llama-3.3-70b-versatile"

print("Client ready.")

Client ready.


In [12]:
## Section 1
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
              temperature=0.7, max_tokens=500):
     response = client.chat.completions.create(
         model=MODEL,
          messages=[
              {"role": "system", "content": system_prompt},
              {"role": "user",   "content": user_prompt},         ],
          temperature=temperature,
          max_tokens=max_tokens,
     )
     return response.choices[0].message.content
#
# TODO: Call it once with a simple question and print the answer.
answer = ask_llm("What is the capital of France?")

# TODO: Print response.usage as well — how many tokens did your call consume?
print("\nAnswer:\n",answer)



Answer:
 The capital of France is Paris.


Student Reasoning - Anatomy of a cell

System against User Roles: The system role establishes  the assitants's persona, overall tone, constraints, and instructions. The user role contains immedaite question.

Tokens and Billing: A token is  a chunck of text. An API  providers bill per token because  processing costs directly reflect the computational memory and compute time required to process prompt context  and generate otput tokens.

In [13]:
##Part 1.2- Temperature: The randomness dial

prompt = "Suggest a name for a savings product for market traders in Accra"
print("=== Temperature = 0.00 ===")




# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
# TODO: Print all 10 answers, grouped by temperature.

for i in range(5):
  print(f"Run{i+1}:", ask_llm(prompt,temperature =0.00, max_tokens = 50 ))

print("\n=== Temperature = 1.20 ====")
for i in range(5):
  print(f"Run{i+1}:", ask_llm(prompt, temperature=1.20, max_tokens=50))



=== Temperature = 0.00 ===
Run1: Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, and this name immediately conveys that the product is tailored to
Run2: Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **
Run3: Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, and this name immediately conveys that the product is tailored to
Run4: Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **
Run5: Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Acc

Student Reasoning- Temperature

Observation: At temperature = 0.00, answers or outputs are identical across all runs. At temperature = 1.20, outputs vary very much, which makes it creative but with possibility of inconsistencies.

Approriate setting: A lower temperature is needed for loan decision support to ensure consistency, predictability and true precision across evaluations.


In [15]:
##Section 2

LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters:
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")


6 letters loaded.


In [16]:
##Section 3
##3.1
# V1: Naive prompt
SUMMARY_PROMPT_V1 = "Summarize this loan application:"

# V2: Structured system + user prompt
SUMMARY_SYSTEM_V2 = "You are an assistant to a microfinance loan officer. Write concise, objective, 3-4 sentence summaries without introducing facts not present in the text."


def summarize_v2(letter):
    return ask_llm(
        user_prompt=f"Summarize this loan application:\n\n{letter}",
        system_prompt=SUMMARY_SYSTEM_V2,
        temperature=0.0,
    )


print("--- V1 Output (L002) ---")
print(ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{LETTERS['L002']}"))

print("\n--- V2 Output (L002) ---")
print(summarize_v2(LETTERS["L002"]))


--- V1 Output (L002) ---
Kwame Boateng, a commercial driver from Kumasi, is applying for a loan of GHS 25,000. He needs the funds to repair his vehicle's engine and pay off personal debts. Although his business has been slow, he expects it to improve after the festive season and is willing to repay the loan as soon as possible. However, he does not have any collateral to secure the loan.

--- V2 Output (L002) ---
Kwame Boateng, a commercial driver in Kumasi, is applying for a GHS 25,000 loan. He needs the funds to repair his trotro engine and settle personal debts. Kwame mentions that business has been slow, but expects it to improve after the festive season. He does not have collateral to offer at the moment.


Student Reasoning — Summarization prompts

V1 vs. V2 Comparison: V1 included conversational filler and unverified assumptions . V2 strictly summarized stated facts within 3 sentences.

No Invented Details: Microfinance decisions impact livelihoods; fabricating facts leads to unfair rejections. This failure mode is termed hallucination.

In [17]:
##3.2
EXTRACT_SYSTEM = """You are a data extraction assistant. Extract details from the loan application into a JSON object matching this exact schema:
{
  "applicant_name": string,
  "amount_ghs": number,
  "purpose": string,
  "monthly_profit_ghs": number or null,
  "has_collateral_or_guarantor": boolean,
  "repayment_months": number or null
}
If a field is not explicitly stated in the letter, set it to null. Do NOT guess or infer values."""

# Few-shot example with fictional letter
FEW_SHOT_USER = """Dear Sir, I am Ama Konadu requesting GHS 5,000 for my bakery in Accra. I can repay in 10 months. My shop has operated for 3 years."""
FEW_SHOT_ASSISTANT = json.dumps(
    {
        "applicant_name": "Ama Konadu",
        "amount_ghs": 5000,
        "purpose": "bakery business",
        "monthly_profit_ghs": None,
        "has_collateral_or_guarantor": False,
        "repayment_months": 10,
    }
)


def extract_fields(letter_text):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": EXTRACT_SYSTEM},
            {"role": "user", "content": FEW_SHOT_USER},
            {"role": "assistant", "content": FEW_SHOT_ASSISTANT},
            {
                "role": "user",
                "content": f"Extract JSON for this letter:\n\n{letter_text}",
            },
        ],
        temperature=0.0,
    )
    raw = response.choices[0].message.content.strip()
    if raw.startswith("```json"):
        raw = raw[7:-3].strip()
    elif raw.startswith("```"):
        raw = raw[3:-3].strip()

    try:
        return json.loads(raw)
    except Exception as e:
        print("JSON parse failure:", e)
        return None


# Extract across all letters
extracted_data = {code: extract_fields(text) for code, text in LETTERS.items()}
df_extracted = pd.DataFrame.from_dict(extracted_data, orient="index")
print(df_extracted)

                          applicant_name  amount_ghs  \
L001                       Akosua Mensah        8000   
L002                       Kwame Boateng       25000   
L003                          Efua Darko       15000   
L004                           Yaw Owusu       12000   
L005  Adenta Women's Weaving Cooperative       30000   
L006                                Kofi       50000   

                                                purpose  monthly_profit_ghs  \
L001    buy a deep freezer and expand into frozen foods               900.0   
L002     repair trotro engine and settle personal debts                 NaN   
L003  purchase industrial sewing machines and fabric...              2800.0   
L004                                       poultry farm              1500.0   
L005                           buy a bulk order of yarn                 NaN   
L006  car washing business, provision shop, and phon...                 NaN   

      has_collateral_or_guarantor  repayment_months  

Student Reasoning — Structured Extraction

Few-shot Example Source: The few-shot example must be distinct from the test set to avoid contamination and ensure evaluation tests true zero-shot or few-shot generalization rather than memorization.

Explicit Null Instruction: Without "use null, do not guess," models often fabricate plausible values for example, hallucinating a profit figure for an unstarted business.

Temperature Choice: temperature=0 maximizes deterministic precision and standard adherence for JSON parsing.

In [18]:
##3.3 Decision support brief

BRIEF_SYSTEM = """You are a decision-support system for a microfinance loan officer.
Analyze the application letter and extracted JSON to produce a brief with:
1. Strengths (bullet points, grounded in text)
2. Risks / Red Flags (bullet points)
3. Missing Information (bullet points)
4. Suggested Next Steps (e.g., "Request bank statements", "Schedule field interview", "Flag for senior review")

Constraint: NEVER output 'Approve' or 'Reject'. Final decisions are made solely by human loan officers."""


def generate_brief(letter_code):
    text = LETTERS[letter_code]
    data = extracted_data[letter_code]
    prompt = f"Letter Text:\n{text}\n\nExtracted Data:\n{json.dumps(data)}"
    return ask_llm(prompt, system_prompt=BRIEF_SYSTEM, temperature=0.0)


for code in ["L001", "L002", "L006"]:
    print(f"\n=== Brief for {code} ===")
    print(generate_brief(code))


=== Brief for L001 ===
**Brief:**

**Strengths:**
* The applicant, Akosua Mensah, has a long-standing business experience of 12 years selling provisions at Makola Market.
* She has a stable monthly profit of GHS 900, which indicates a consistent income stream.
* Akosua has demonstrated a good savings habit through the susu scheme, saving GHS 2,500 over two years without missing a contribution.
* She has a guarantor, her sister, who is a teacher, which provides an added layer of security for the loan.
* The applicant has a clear plan for using the loan, which is to buy a deep freezer and expand into frozen foods.

**Risks / Red Flags:**
* The loan amount of GHS 8,000 is significant compared to the applicant's monthly profit, which may pose a repayment risk if the business does not generate enough income.
* The expansion into frozen foods may require additional investments or pose new operational risks that are not explicitly addressed in the application.
* There is no information on th

Student Reasoning — Decision Support

L003 vs L006: The system correctly identifies L003's strong revenue history and fixed deposit collateral as strengths, while flagging L006's lack of operating history, speculative multi-industry focus, and zero collateral.

Forbidding "Approve"/"Reject":

Practical: LLMs lack access to real-time physical verification, site visits, and context required for credit decisions.

Ethical: Keeping humans in the loop prevents algorithmic bias, maintains legal accountability, and protects applicant rights.